In [ ]:
import os
import re
import subprocess

# --- 1. SETUP ---
cha_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2334_T4.cha"
video_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2334_T4.mp4"
output_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2"

os.makedirs(output_dir, exist_ok=True)

# --- 2. REGEX PATTERNS ---
# This strictly looks for the timestamp bullets: 1917578_1923343
timestamp_pattern = re.compile(r"(\d+)_(\d+)")

clip_count = 0
print(f"🎧 Scanning {cha_file} and extracting 22050Hz audio...")

# --- 3. PARSING & EXTRACTION ---
with open(cha_file, "r", encoding="utf-8") as f:
    for line in f:
        # ONLY look at the Patient's lines
        if line.startswith("*PAR:"):
            match = timestamp_pattern.search(line)
            
            if match:
                start_ms = int(match.group(1))
                end_ms = int(match.group(2))
                
                # Convert to seconds
                start_sec = start_ms / 1000.0
                duration_sec = (end_ms - start_ms) / 1000.0
                
                # --- CLEAN THE TRANSCRIPT ---
                # Remove the '*PAR:' prefix and the timestamp bullet to isolate the text
                clean_text = line.replace("*PAR:", "").split("")[0].strip()
                
                # Skip completely empty lines just in case
                if not clean_text:
                    continue

                # --- 4. FILE PATHS ---
                base_filename = f"patient1_{clip_count:04d}"
                out_wav = os.path.join(output_dir, f"{base_filename}.wav")
                out_txt = os.path.join(output_dir, f"{base_filename}.txt")
                
                # --- 5. SAVE THE TRANSCRIPT ---
                with open(out_txt, "w", encoding="utf-8") as txt_file:
                    txt_file.write(clean_text)
                
                # --- 6. CROP & CONVERT AUDIO WITH FFMPEG ---
                cmd = [
                    "ffmpeg", "-y", 
                    "-ss", str(start_sec), 
                    "-i", video_file, 
                    "-t", str(duration_sec),
                    "-ar", "22050",          # Resample directly to 22050 Hz
                    "-ac", "1",              # Convert to Mono audio
                    "-c:a", "pcm_s16le",     # Uncompressed 16-bit WAV format
                    "-vn",                   # Strip the video track completely
                    out_wav
                ]
                
                # Run FFmpeg silently
                subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                
                clip_count += 1

print(f"\n✅ Done! Extracted {clip_count} audio files and transcripts to the '{output_dir}' folder.")